The "Interpreter Breaker" Set
1. The Late Binding Trap (The Closure Boss)
This is arguably the most famous Python interview question in existence. What does this print?

In [2]:
funcs = []
for i in range(3):
    funcs.append(lambda: i)

for f in funcs:
    print(f(), end=" ")

2 2 2 

2. Schrödinger's Tuple (Testing your += knowledge)
We just learned how += has a "Read Phase" and a "Write Phase." What happens here?

In [3]:
t = ([1, 2], 3)

try:
    t[0] += [4, 5]
except TypeError:
    print("Caught an error!")

print(t)

Caught an error!
([1, 2, 4, 5], 3)


In [ ]:
[1, 2] + [4, 5]

[1, 2, 4, 5]

In [ ]:
l1 = [1, 2]
l2 = [3, 4]
l3 = l1 + l2
print(l1, l2, l3)

[1, 2] [3, 4] [1, 2, 3, 4]


In [ ]:
l1, l2

([1, 2], [3, 4])

In [28]:
l1 += l2
# equivalent to .extend()

In [ ]:
t = ([0, 1], 2, 3)
t[0].append(2)
t[0].extend([3, 4])
print(t)
# this works properly, but  =  does not work because it tries to reassign the tuple element, which is not allowed.

([0, 1, 2, 3, 4], 2, 3)


In [ ]:
l1, l2

([1, 2, 3, 4], [3, 4])

In [ ]:
l1 = [1, 2, 3]
result = l1.extend([4, 5, 6])
print(result)
print(l1)

None
[1, 2, 3, 4, 5, 6]


 If you just run a normal list like my_list += [4, 5], Python executes my_list = my_list.__iadd__([4, 5]). It reassigns the list to itself. It seems silly because the list was already changed in place!But it is not a bug. It is designed this way because the += operator has to work for both mutable objects (lists) and immutable objects (strings/integers).The Universal Rule of +=The Python interpreter uses the exact same mechanics for everything when handling +=. It always expects a return value from the right side to assign to the left side:$$\text{Variable} = \text{Variable}.\_\_\text{iadd}\_\_(\text{Value})$$Let's look at how this single design choice forces two different behaviors:1. Immutable Objects (Like Strings)Strings cannot be changed in place. So __iadd__ on a string must create a brand-new string object and return it so it can be assigned to the variable name.

3. The Object-Oriented Mutator
This tests if you truly understand the difference between creating an instance attribute with = versus mutating a class attribute with .append().

In [35]:
s = "Hello"
s += " World"  # s = s.__iadd__(" World")
# Returns a BRAND NEW string object: "Hello World"

2. Mutable Objects (Like Lists)
Lists can be changed in place. But because Python uses the exact same architecture for +=, the list's __iadd__ method is forced by the language rules to return something to complete the assignment step.

To prevent breaking the chain, the creators of Python made it return self (the list itself).

Why the Tuple Crashes
Because of this universal design, when you write t[0] += [4, 5], Python blindly follows the master formula:

In [36]:
t[0] = t[0].__iadd__([4, 5])

TypeError: 'tuple' object does not support item assignment

t[0].__iadd__([4, 5]) executes. It mutates the list inside the heap, and returns the list's memory address (self).

Python then tries to execute the t[0] = ... assignment step with that returned address.

The tuple drops the hammer and says, "No assignments allowed!"

It feels like a bug because the list mutation step sneaks through before the tuple's assignment block can stop it. It is the ultimate consequence of Python prioritizing consistent internal mechanics over edge-case safety.

In [ ]:
class Data:
    items = []


d1 = Data()
d2 = Data()

d1.items.append(1)
d2.items = [2]
d1.items.append(3)

print(f"Class: {Data.items}")
print(f"d1: {d1.items}")
print(f"d2: {d2.items}")

Class: [1, 3]
d1: [1, 3]
d2: [2]


In [8]:
print(f"Class: {id(Data.items)}")
print(f"d1: {id(d1.items)}")
print(f"d2: {id(d2.items)}")

Class: 2999910852864
d1: 2999910852864
d2: 2999910853056


4. The Compilation vs Execution Trap
Remember how Python decides what is a local variable before the code runs? What happens when we try to catch it?

In [ ]:
x = 10


def shadow_test():
    try:
        print(x)
    except UnboundLocalError:
        print("Caught the UnboundLocalError!")

    x = 20
    print(x)


shadow_test()

Caught the UnboundLocalError!
20


5. The Scope Disconnect
Does changing the global variable change the closure's memory?

In [ ]:
def create_multiplier(factor):
    def multiplier(number):
        return factor * number

    return multiplier


factor = 10
mult_by_2 = create_multiplier(2)

# I changed the global variable right before running the function!
factor = 100

print(mult_by_2(5))

10


The "Global Mutation" Test
Do not run this yet. Trace the memory. We are passing in one global, referencing another directly, and then mutating both before running the closure.

In [11]:
global_status = "Active"
global_data = [10, 20]


def create_dashboard(passed_data):
    def render():
        # Referencing a global variable directly (not passed in)
        print(f"Status: {global_status}")

        # Referencing the passed argument (local to the outer function)
        print(f"Data: {passed_data}")

    return render


# We create the closure and pass the global list into it
app_render = create_dashboard(global_data)

# NOW we change the globals BEFORE running the inner function
global_status = "Offline"
global_data.append(30)
global_data = [999]  # Pay close attention to this line!

# What prints here?
app_render()

Status: Offline
Data: [10, 20, 30]


In [ ]:
global_status = "Active"


def create_closure():
    def render():
        print(global_status)

    return render


app_render = create_closure()

# We delete the global variable from the dictionary
del global_status

# What happens?
app_render()

NameError: name 'global_status' is not defined

The Result: NameError: name 'global_status' is not defined

The Why: Late binding means the closure doesn't memorize "Active". It memorizes the instruction: "Go to the global dictionary and look up the key global_status." Because del removes that key from the global dictionary, the closure looks for it, finds nothing, and crashes.

Scenario 2: Passed Pointer (Local Variable) -> SURVIVES
If you pass the global variable into the outer function, it survives the deletion.

In [13]:
global_data = [10, 20]


def create_closure(passed_data):
    def render():
        print(passed_data)

    return render


# We pass the global in. passed_data now points to [10, 20] in the Heap
app_render = create_closure(global_data)

# We delete the global name
del global_data

# What happens?
app_render()

[10, 20]


The Result: [10, 20]

The Why: del does not delete objects from the Heap memory. It only deletes the name (the tag) from the namespace.

global_data and passed_data were both pointing to the same list in the Heap.

del global_data destroys the global_data tag.

However, passed_data (living safely inside the closure's memory) is still holding tightly to that list in the Heap.

Python's Garbage Collector sees that the list is still being used by passed_data, so it leaves the memory alone.

In [14]:
global_data = [10, 20]


def create_closure():
    def render():
        print(global_data)

    return render


# We pass the global in. passed_data now points to [10, 20] in the Heap
app_render = create_closure()

# We delete the global name
del global_data

# What happens?
app_render()

NameError: name 'global_data' is not defined

In [ ]:
global_list = [10, 20]


def create_closure():
    def render():
        print(global_list)  # Direct reference, no arguments passed

    return render


app_render = create_closure()

# 1. We MUTATE it. What happens?
global_list.append(30)
app_render()  # Output: [10, 20, 30]

# 2. We DELETE it. What happens?
del global_list
app_render()  # Output: NameError: name 'global_list' is not defined

[10, 20, 30]


NameError: name 'global_list' is not defined

The "Why"
Because you didn't pass it as an argument, the closure has no local pointer protecting the memory in the Heap.

Instead, the closure wrote down a strict instruction: "When I run, go to the Global Dictionary and look for a tag named global_list."

When you mutate it (append): The closure goes to the dictionary, finds the tag, follows it to the Heap, and prints the updated list.

When you delete it (del): The del command destroys the name tag in the Global Dictionary. When the closure runs, it checks the dictionary, finds no tag named global_list, and panics.

It does not matter if the object is mutable (a list) or immutable (a string). If it is a direct reference, the closure is completely at the mercy of the global namespace dictionary. If the name dies, the closure crashes.

In [17]:
# Our Global Variables
total_sales = 100  # Immutable (Integer)
recent_items = ["Milk"]  # Mutable (List)


def create_register():
    def process_transaction():
        # Case A: Trying to UPDATE a global integer without 'global'
        try:
            total_sales += 50
        except UnboundLocalError:
            print("Crash A: Cannot update total_sales!")

        # Case B: Trying to APPEND to a global list without 'global'
        recent_items.append("Bread")
        print(f"Success B: {recent_items}")

    return process_transaction


register = create_register()
register()
print(recent_items)

Crash A: Cannot update total_sales!
Success B: ['Milk', 'Bread']
['Milk', 'Bread']


In [18]:
x = 10  # 1. Initial global state


def create_closure():
    def inner():
        global x
        # 4. Inner function runs. What does it see?
        print(f"Inner starting value: {x}")

        # 5. Inner function updates the global
        x = 999
        print(f"Inner changed it to: {x}")

    return inner


# 2. Create the function
my_func = create_closure()

# 3. Change the global OUTSIDE, before the function runs
x = 50
print(f"Outside changed it to: {x}")

# Call the function
my_func()

# 6. Check the global OUTSIDE after the function runs
print(f"Outside final check: {x}")

Outside changed it to: 50
Inner starting value: 50
Inner changed it to: 999
Outside final check: 999


The "Why" (How the Compiler Sees It)
When you use the global keyword inside an inner function, you completely bypass the concept of a closure for that variable.

No Protected Memory: The inner function does not take a snapshot of x. It does not create a local pointer to x.

The Direct Pipeline: By writing global x, inner() builds a direct, hardwired pipe straight to the Global Dictionary.

Two-Way Street: * When the outside world changes x to 50, the inner function sees 50 because it just looks at the global dictionary.

When the inner function changes x to 999, the outside world sees 999 because the inner function directly overwrote the global dictionary.

There is no "scoping clash" or "late binding" trickery here. They are both literally holding the same TV remote, pressing buttons on the exact same TV.

In [19]:
def my_func():
    # 'future_var' does not exist yet!
    # Python doesn't care at definition time.
    print(future_var)


# We create it LATER.
future_var = "I exist now!"

# The function accesses it at execution time. It works perfectly.
my_func()

I exist now!


In [ ]:
# 1. Our global variable
global_box = ["Item_A"]


def create_closure(saved_box):
    # 'saved_box' is now locked into the closure's memory
    def inner():
        print(f"Inner starts with: {saved_box}")

        # 4. Inner function updates the data
        saved_box.append("Added_Inside")
        print(f"Inner ends with:   {saved_box}")

    return inner


# 2. Create the closure (passing the global pointer in)
my_func = create_closure(global_box)

# 3. Change it OUTSIDE BEFORE calling the inner function
global_box.append("Added_Before")
print(f"Outside before call: {global_box}\n")

# Call the inner function
my_func()

# 5. Change it OUTSIDE AFTER calling the inner function
global_box.append("Added_After")
print(f"\nOutside after call:  {global_box}")

# 6. Call the inner function one last time to prove the link!
print("\n--- Final Proof ---")
my_func()

Outside before call: ['Item_A', 'Added_Before']

Inner starts with: ['Item_A', 'Added_Before']
Inner ends with:   ['Item_A', 'Added_Before', 'Added_Inside']

Outside after call:  ['Item_A', 'Added_Before', 'Added_Inside', 'Added_After']

--- Final Proof ---
Inner starts with: ['Item_A', 'Added_Before', 'Added_Inside', 'Added_After']
Inner ends with:   ['Item_A', 'Added_Before', 'Added_Inside', 'Added_After', 'Added_Inside']


In [ ]:
x = 10  # The Global


def outer(passed_x):
    # 'passed_x' is now a pure local variable in outer's scope!

    def inner():
        nonlocal passed_x  # Success! It finds 'passed_x'
        passed_x += 999  # Updates the closure's memory
        print(f"Inside closure: {passed_x}")

    # We return inner, plus a quick lambda to peek at the closure's memory later
    return inner, lambda: passed_x


my_inner, check_closure = outer(x)

my_inner()
print(f"Closure memory now holds: {check_closure()}")
print(f"But global x is still: {x}")

Inside closure: 1009
Closure memory now holds: 1009
But global x is still: 10


The Catch
You successfully created the closure and updated it with nonlocal, but notice that the original global x is still 10.

Because integers are immutable, when you wrote passed_x = 999, you just pointed the closure's local passed_x tag to a brand new number in the Heap. The global x tag was left behind, pointing at the original 10.

In [23]:
my_inner()
print(f"Closure memory now holds: {check_closure()}")
print(f"But global x is still: {x}")

Inside closure: 2008
Closure memory now holds: 2008
But global x is still: 10


and that both retunred functions's passed value will be same always right?

You are absolutely, 100% correct.

This is a concept called a Shared Closure, and your intuition is spot on. They will always be perfectly synced because they are looking at the exact same physical space in memory.

The "Single Box" Rule
When you call outer(x), Python does not create a separate memory space for inner and another separate space for the lambda.

Instead, it does this:

outer runs and creates a single, hidden memory box for passed_x.

Python defines inner and hands it the key to that box.

Python defines the lambda and hands it a copy of the exact same key.

Because they both hold the key to the same box, if inner reaches in and changes the value to 999, the lambda will immediately see 999 the next time it looks inside.

In [ ]:
# Creates Box #1 (Shared by these two functions)
inner_A, check_A = outer(10)

# Creates Box #2 (Shared by these two functions)
inner_B, check_B = outer(50)

# Run inner_A to change Box #1 to 999
inner_A()

print(check_A())  # Prints 999 (It saw the change!)
print(check_B())  # Prints 50 (Completely unaffected. It looks at Box #2)

Inside closure: 1009
1009
50
